# Task 5 v4 — Xử lý trọng số `body_part` bị lấn át (theo 3 đề xuất từ kết quả v3)

**Bối cảnh từ `kmeans_v3.ipynb` (đã chạy, có số liệu thật):**
1. `distance_to_goal` chỉ lệch nhẹ (skew=0.448, gần đối xứng) — `log1p` không giúp gì, còn làm
   skew nặng hơn theo chiều ngược lại (-0.594). **Kết luận: bỏ log1p** — notebook này không dùng.
2. Bảng centroid `RobustScaler, k=5` cho thấy **3/5 cụm gần như là bản sao thuần tuý của
   `body_part`** (cụm ~100% chân phải, cụm ~100% chân trái, cụm ~93% đánh đầu) — do one-hot
   `body_part` chiếm 4/9 chiều feature, phóng đại trọng số của nó trong khoảng cách L2 so với các
   feature liên tục (distance/angle/pressure).
3. Silhouette-max mắc kẹt ở k=2 bất kể có log hay không — cảnh báo: **đừng chỉ tối ưu số**, phải
   nhìn centroid + trực quan để biết cụm có ý nghĩa chiến thuật hay chỉ đang lặp lại thông tin
   `body_part` đã biết trước.

**3 nhánh so sánh trong notebook này** (đều dựa trên `RobustScaler`, không log, đã lọc ADR-004
giống `kmeans_v3.ipynb`):

| Nhánh | Ý tưởng |
|---|---|
| `Baseline (co body_part)` | Y hệt nhánh RobustScaler của v3 — làm đối chứng |
| `Khong body_part` | Bỏ hẳn 4 cột one-hot `body_part` khỏi feature cluster — chỉ dùng để phân tích hậu-kỳ (post-hoc) trên centroid |
| `Body_part giam trong so` | Vẫn giữ `body_part` trong X, nhưng nhân 4 cột đó với hệ số `< 1` **sau khi scale**, để giảm (không xoá hẳn) ảnh hưởng lên khoảng cách L2 |

Và một **thước đo mới** — "độ thuần `body_part`" của cụm — để kiểm chứng bằng số xem 2 cách sửa
trên có thực sự giải quyết được vấn đề "cụm = bản sao body_part" hay không, thay vì chỉ nhìn
silhouette (đúng đề xuất 3).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

DATA_DIR = "/kaggle/working"  # doi lai neu chay o noi khac

RAW_CSV_CANDIDATES = [
    f"{DATA_DIR}/shots_worldcup_2018_2022_raw.csv",
    f"{DATA_DIR}/extract_feature_worldcup_2018_2022_raw.csv",
    "shots_worldcup_2018_2022_raw.csv",
    "extract_feature_worldcup_2018_2022_raw.csv",
    "/kaggle/input/datasets/tunlcvcng/raw-wc2018-2022/extract_feature_worldcup_2018_2022_raw.csv",
]

df_raw = None
used_path = None
for candidate in RAW_CSV_CANDIDATES:
    try:
        df_raw = pd.read_csv(candidate, encoding="utf-8-sig")
        used_path = candidate
        break
    except FileNotFoundError:
        continue

if df_raw is None:
    raise FileNotFoundError(
        f"Khong tim thay file raw shot nao trong: {RAW_CSV_CANDIDATES}. "
        "Sua RAW_CSV_CANDIDATES cho dung ten/duong dan file ban dang co tren Kaggle."
    )

print(f"Da nap: {used_path} | shape = {df_raw.shape}")
df_raw.head(3)

## 5v4.1 — Lọc penalty / luân lưu (ADR-004) — giống `kmeans_v3.ipynb`

In [ ]:
EXCLUDE_SHOT_TYPES = ("Penalty",)
EXCLUDE_PERIODS = (5,)  # luat luan luu

n_before = len(df_raw)
mask_penalty = df_raw["shot_type"].isin(EXCLUDE_SHOT_TYPES)
mask_shootout = df_raw["period"].isin(EXCLUDE_PERIODS)
mask_exclude = mask_penalty | mask_shootout

print(f"Shot bi loai vi shot_type in {EXCLUDE_SHOT_TYPES}: {int(mask_penalty.sum())}")
print(f"Shot bi loai vi period in {EXCLUDE_PERIODS} (luan luu): {int(mask_shootout.sum())}")

df = df_raw.loc[~mask_exclude].reset_index(drop=True)
print(f"Tong shot truoc loc: {n_before} -> sau loc: {len(df)} (da loai {n_before - len(df)})")

## 5v4.2 — Xây dựng feature matrix

Tách riêng `BASE_FEATURE_COLS` (liên tục/nhị phân theo tình huống) và `body_part_cols` (one-hot
kỹ thuật dứt điểm) — để dễ bật/tắt/giảm trọng số body_part ở bước sau mà không đổi phần còn lại.

In [ ]:
BASE_FEATURE_COLS = [
    "distance_to_goal", "angle_to_goal", "n_opponents_in_frame", "n_teammates_in_frame", "under_pressure",
]

df_encoded = pd.get_dummies(df, columns=["body_part"], drop_first=False)
body_part_cols = sorted([c for c in df_encoded.columns if c.startswith("body_part_")])
print("Cac cot body_part sau one-hot:", body_part_cols)

FEATURE_COLS = BASE_FEATURE_COLS + body_part_cols

df_encoded["under_pressure"] = df_encoded["under_pressure"].astype(int)
for c in body_part_cols:
    df_encoded[c] = df_encoded[c].astype(int)

X_unscaled_features = df_encoded[FEATURE_COLS].copy()

n_missing = X_unscaled_features.isna().sum()
print("\nMissing con lai (can xu ly truoc khi scale):")
print(n_missing[n_missing > 0] if n_missing.sum() > 0 else "khong co")

print("\nX_unscaled_features shape:", X_unscaled_features.shape)
print("FEATURE_COLS:", FEATURE_COLS)

## 5v4.3 — Ba nhánh so sánh

`BODY_PART_WEIGHT` là hệ số nhân 4 cột `body_part_*` **sau khi** đã `RobustScaler` toàn bộ 9
cột — đặt 0.3 làm mặc định (giảm ~70% đóng góp của body_part vào khoảng cách L2), có thể chỉnh
lại và chạy so sánh thêm nếu muốn dò giá trị tốt hơn.

In [ ]:
BODY_PART_WEIGHT = 0.3  # 0 = bo hoan toan (giong nhanh "Khong body_part"), 1 = giu nguyen (giong Baseline)

# --- Nhanh 1: Baseline (co body_part, khong giam trong so) ---
X_baseline = RobustScaler().fit_transform(X_unscaled_features)

# --- Nhanh 2: Khong body_part (bo han 4 cot one-hot khoi feature cluster) ---
X_no_bp = RobustScaler().fit_transform(X_unscaled_features[BASE_FEATURE_COLS])

# --- Nhanh 3: Body_part giam trong so (giu nhung nhan he so < 1 sau khi scale) ---
X_weighted = X_baseline.copy()
body_part_idx = [X_unscaled_features.columns.get_loc(c) for c in body_part_cols]
X_weighted[:, body_part_idx] = X_weighted[:, body_part_idx] * BODY_PART_WEIGHT

X_versions = {
    "Baseline (co body_part)": X_baseline,
    "Khong body_part": X_no_bp,
    "Body_part giam trong so": X_weighted,
}

for name, X in X_versions.items():
    print(name, X.shape)

## 5v4.4 — Elbow Method (kneedle, song song 3 nhánh)

In [ ]:
K_RANGE = list(range(2, 11))
colors = {
    "Baseline (co body_part)": "#c0392b",
    "Khong body_part": "#2980b9",
    "Body_part giam trong so": "#27ae60",
}


def kneedle_elbow(k_values, values):
    k = np.array(k_values, dtype=float)
    y = np.array(values, dtype=float)
    k_norm = (k - k.min()) / (k.max() - k.min())
    y_norm = (y - y.min()) / (y.max() - y.min())
    x1, y1 = k_norm[0], y_norm[0]
    x2, y2 = k_norm[-1], y_norm[-1]
    num = np.abs((y2 - y1) * k_norm - (x2 - x1) * y_norm + x2 * y1 - y2 * x1)
    den = np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)
    dist = num / den
    idx = np.argmax(dist)
    return int(k[idx])


inertia_results = {}
for name, X in X_versions.items():
    inertias = []
    for k in K_RANGE:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        km.fit(X)
        inertias.append(km.inertia_)
    inertia_results[name] = inertias

elbow_k = {name: kneedle_elbow(K_RANGE, vals) for name, vals in inertia_results.items()}
print("Elbow k (kneedle) theo tung nhanh:", elbow_k)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharex=True)
for ax, name in zip(axes, X_versions.keys()):
    ax.plot(K_RANGE, inertia_results[name], marker="o", color=colors[name])
    ax.axvline(elbow_k[name], linestyle="--", color="gray", label=f"elbow k={elbow_k[name]}")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("k")
    ax.set_ylabel("Inertia (WCSS)")
    ax.legend()

plt.suptitle("Elbow Method — 3 nhanh (khong so gia tri inertia tuyet doi giua cac nhanh)", y=1.03)
plt.tight_layout()
plt.show()

## 5v4.5 — Silhouette Score (song song 3 nhánh)

In [ ]:
silhouette_results = {}
for name, X in X_versions.items():
    sils = []
    for k in K_RANGE:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(X)
        sils.append(silhouette_score(X, labels))
    silhouette_results[name] = sils

best_k_silhouette = {name: K_RANGE[int(np.argmax(vals))] for name, vals in silhouette_results.items()}
print("Best k theo Silhouette:", best_k_silhouette)

fig, ax = plt.subplots(figsize=(9, 5))
for name, vals in silhouette_results.items():
    ax.plot(K_RANGE, vals, marker="o", label=name, color=colors[name])
ax.set_xlabel("k")
ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score theo k — 3 nhanh")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5v4.6 — Gap Statistic (Tibshirani, song song 3 nhánh)

In [ ]:
def compute_gap_statistic(X, k_range, B=10, random_state=42):
    rng = np.random.default_rng(random_state)
    mins, maxs = X.min(axis=0), X.max(axis=0)

    def fit_inertia(data, k, seed):
        km = KMeans(n_clusters=k, n_init=10, random_state=seed)
        km.fit(data)
        return km.inertia_

    gaps, sks = [], []
    for k in k_range:
        wk_real = fit_inertia(X, k, random_state)

        log_wk_refs = []
        for b in range(B):
            X_ref = rng.uniform(low=mins, high=maxs, size=X.shape)
            wk_ref = fit_inertia(X_ref, k, random_state + b + 1)
            log_wk_refs.append(np.log(wk_ref))
        log_wk_refs = np.array(log_wk_refs)

        gap = log_wk_refs.mean() - np.log(wk_real)
        sk = log_wk_refs.std() * np.sqrt(1 + 1.0 / B)
        gaps.append(gap)
        sks.append(sk)

    return np.array(gaps), np.array(sks)


def choose_k_tibshirani(k_range, gaps, sks):
    for i in range(len(k_range) - 1):
        if gaps[i] >= gaps[i + 1] - sks[i + 1]:
            return k_range[i]
    return None  # khong tim duoc k thoa dieu kien trong pham vi da thu


gap_results = {}
for name, X in X_versions.items():
    gaps, sks = compute_gap_statistic(X, K_RANGE, B=10, random_state=42)
    gap_results[name] = (gaps, sks)

gap_k = {name: choose_k_tibshirani(K_RANGE, gaps, sks) for name, (gaps, sks) in gap_results.items()}
print("k chon theo Gap Statistic (quy tac Tibshirani):", gap_k)

fig, ax = plt.subplots(figsize=(9, 5))
for name, (gaps, sks) in gap_results.items():
    ax.errorbar(K_RANGE, gaps, yerr=sks, marker="o", capsize=3, label=name, color=colors[name])
ax.set_xlabel("k")
ax.set_ylabel("Gap Statistic")
ax.set_title("Gap Statistic theo k — 3 nhanh (thanh loi = s(k))")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5v4.7 — Bảng tổng hợp k đề xuất

In [ ]:
summary_k = pd.DataFrame({
    "Elbow (kneedle)": elbow_k,
    "Silhouette (max)": best_k_silhouette,
    "Gap Statistic (Tibshirani)": gap_k,
})
summary_k

## 5v4.8 — "Độ thuần body_part" của cụm — kiểm chứng bằng số (đề xuất 3: đừng chỉ nhìn silhouette)

Với mỗi cụm, tính tỉ lệ lớn nhất trong 4 nhóm `body_part` (Head/Left Foot/Other/Right Foot).
Lấy trung bình có trọng số theo cỡ cụm ⇒ **"độ thuần body_part" trung bình toàn bộ phân cụm**.

- Gần **1.0** ⇒ cụm gần như là bản sao `body_part` (đúng vấn đề đã thấy ở `kmeans_v3.ipynb`, k=5:
  baseline ước tính quanh **0.9+** — 3/5 cụm gần thuần 100%).
- Càng **thấp** (tiến gần tỉ lệ nền tự nhiên — vd 4 nhóm chia đều thì baseline ngẫu nhiên ~25-50%
  tuỳ độ mất cân bằng thật của `body_part`) ⇒ cụm đang được chia theo bối cảnh (distance/pressure/
  crowd) chứ không phải theo kỹ thuật dứt điểm — đúng mục tiêu ban đầu.

In [ ]:
def body_part_purity(labels, body_part_df=df_encoded[body_part_cols]):
    tmp = body_part_df.copy()
    tmp["Cluster"] = labels
    proportions = tmp.groupby("Cluster").mean()  # ty le tung body_part trong moi cum
    per_cluster_purity = proportions.max(axis=1)  # ty le lon nhat trong moi cum
    cluster_sizes = tmp["Cluster"].value_counts().sort_index()
    weighted_purity = (per_cluster_purity * cluster_sizes).sum() / cluster_sizes.sum()
    return weighted_purity, per_cluster_purity


# Dung chung 1 gia tri k (uu tien k trung nhieu phuong phap nhat o bang 5v4.7, doi lai neu can)
PURITY_CHECK_K = 5

purity_summary = []
for name, X in X_versions.items():
    km = KMeans(n_clusters=PURITY_CHECK_K, n_init=10, random_state=42)
    lbl = km.fit_predict(X)
    weighted_purity, per_cluster = body_part_purity(lbl)
    purity_summary.append({"nhanh": name, "body_part_purity (k=%d)" % PURITY_CHECK_K: weighted_purity})
    print(f"--- {name} (k={PURITY_CHECK_K}) ---")
    print("Purity tung cum:", per_cluster.round(3).to_dict())
    print(f"Purity trung binh (weighted theo co cum): {weighted_purity:.3f}\n")

purity_df = pd.DataFrame(purity_summary).set_index("nhanh")
display(purity_df)

## 5v4.9 — ARI giữa 3 nhánh theo k

So khớp cấu trúc cụm giữa 3 nhánh — ARI thấp giữa Baseline và 2 nhánh còn lại nghĩa là việc bỏ/giảm
trọng số `body_part` thực sự đổi cách chia cụm (đúng kỳ vọng), không chỉ đổi centroid vặt.

In [ ]:
ari_k_range = [2, 3, 4, 5, 6, 7]
ari_rows = []
cluster_labels_by_k = {}

names = list(X_versions.keys())
for k in ari_k_range:
    labels = {}
    for name, X in X_versions.items():
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels[name] = km.fit_predict(X)
    cluster_labels_by_k[k] = labels

    row = {"k": k}
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            ari = adjusted_rand_score(labels[names[i]], labels[names[j]])
            row[f"ARI({names[i]}, {names[j]})"] = ari
    ari_rows.append(row)

ari_df = pd.DataFrame(ari_rows).set_index("k")
display(ari_df)

fig, ax = plt.subplots(figsize=(10, 5))
for col in ari_df.columns:
    ax.plot(ari_df.index, ari_df[col], marker="o", label=col)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("k")
ax.set_ylabel("Adjusted Rand Index")
ax.set_title("ARI giua 3 nhanh theo k")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5v4.10 — Chốt phương án & trực quan hoá không gian

`FINAL_K` đặt mặc định = 5 để so trực tiếp với `kmeans_v2.ipynb`/`kmeans_v3.ipynb` — đổi lại
theo bảng 5v4.7 nếu kết quả thực tế gợi ý k khác.

In [ ]:
FINAL_K = 5  # doi lai neu bang 5v4.7 goi y k khac

labels_at_final_k = cluster_labels_by_k.get(FINAL_K)
if labels_at_final_k is None:
    labels_at_final_k = {}
    for name, X in X_versions.items():
        km = KMeans(n_clusters=FINAL_K, n_init=10, random_state=42)
        labels_at_final_k[name] = km.fit_predict(X)

# Centroid luon mo ta bang feature GOC (chua scale/giam trong so) de de doc/bao cao,
# kem location_x/y de dung cho bieu do dia ly o cell sau.
describe_cols = FEATURE_COLS + ["location_x", "location_y"]
unscaled_features_original = df_encoded[describe_cols].copy()

centroids_original = {}
for name, lbl in labels_at_final_k.items():
    tmp = unscaled_features_original.copy()
    tmp["Cluster"] = lbl
    centroids_original[name] = tmp.groupby("Cluster").mean()
    _, per_cluster_purity = body_part_purity(lbl)
    print(f"=== {name} (k={FINAL_K}) - so shot / cum ===")
    print(pd.Series(lbl).value_counts().sort_index().to_dict())
    print("Purity tung cum:", per_cluster_purity.round(3).to_dict())
    print()

print(f"--- Centroid (Baseline (co body_part), k={FINAL_K}) ---")
display(centroids_original["Baseline (co body_part)"])
print(f"\n--- Centroid (Khong body_part, k={FINAL_K}) ---")
display(centroids_original["Khong body_part"])
print(f"\n--- Centroid (Body_part giam trong so, k={FINAL_K}) ---")
display(centroids_original["Body_part giam trong so"])

In [ ]:
# Bieu do phan bo vi tri cu sut theo cum, 3 nhanh canh nhau
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle(f"Geographical Distribution of Shot Clusters — 3 nhanh (k={FINAL_K})", fontsize=14)

for ax, name in zip(axes, X_versions.keys()):
    lbl = labels_at_final_k[name]

    sns.scatterplot(
        x=df["location_x"], y=df["location_y"],
        hue=lbl, palette="tab10", ax=ax, s=20, alpha=0.4,
    )
    ax.scatter(
        x=centroids_original[name]["location_x"], y=centroids_original[name]["location_y"],
        color="black", marker="*", s=400, edgecolor="white", linewidth=1.5, label="Centroid",
    )
    ax.plot([0, 120, 120, 0, 0], [0, 0, 80, 80, 0], color="black")
    ax.plot([120, 120], [36, 44], color="red", linewidth=4)
    ax.set_title(f"{name} (k={FINAL_K})", fontsize=10)
    ax.set_xlabel("location_x")
    ax.set_ylabel("location_y")
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=7)

plt.tight_layout()
plt.show()

## 5v4.11 — PCA 2D: đánh giá "công bằng" hơn thay biểu đồ vị trí

Biểu đồ `location_x/y` ở trên chỉ chiếu đúng 2 chiều, mà **5/9 feature dùng để cluster**
(`n_opponents_in_frame`, `n_teammates_in_frame`, `under_pressure`, 4 cột `body_part`) **không hề
ràng buộc với vị trí sút** — hai shot cùng một điểm trên sân hoàn toàn có thể khác cụm, và cùng
một cụm có thể rải khắp sân. Chồng lấn trên biểu đồ đó không chứng minh được cụm "không hiệu quả".

PCA chiếu đúng không gian **mà K-Means thực sự nhìn thấy** (`X_versions[...]`, đã scale) xuống 2
chiều giữ nhiều phương sai nhất — công bằng hơn để đánh giá mức độ tách cụm thật sự.

⚠️ Đây vẫn là một phép chiếu **mất thông tin** — % phương sai PC1+PC2 giữ được in ngay trên mỗi
subplot. Nếu con số đó thấp (dưới ~50%), chồng lấn màu trên hình 2D vẫn **không** chứng minh được
cụm chồng lấn thật trong không gian gốc nhiều chiều — chỉ nên đọc như một gợi ý trực quan, không
phải bằng chứng cuối cùng (silhouette/ARI/purity ở các mục trước mới là số liệu chính xác).

In [ ]:
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle(f"PCA 2D — khong gian feature thuc su dung de cluster (k={FINAL_K})", fontsize=14)

for ax, name in zip(axes, X_versions.keys()):
    X = X_versions[name]
    lbl = labels_at_final_k[name]

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)
    evr = pca.explained_variance_ratio_

    sns.scatterplot(
        x=X_pca[:, 0], y=X_pca[:, 1],
        hue=lbl, palette="tab10", ax=ax, s=20, alpha=0.5,
    )

    # Centroid trong khong gian PCA = trung binh cac diem da chieu theo tung cum
    # (tuong duong PCA cua centroid goc, vi PCA la phep chieu tuyen tinh)
    pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
    pca_df["Cluster"] = lbl
    pca_centroids = pca_df.groupby("Cluster")[["PC1", "PC2"]].mean()
    ax.scatter(
        pca_centroids["PC1"], pca_centroids["PC2"],
        color="black", marker="*", s=400, edgecolor="white", linewidth=1.5, label="Centroid",
    )

    ax.set_title(f"{name}\nPC1+PC2 giai thich {evr.sum():.1%} phuong sai", fontsize=10)
    ax.set_xlabel(f"PC1 ({evr[0]:.1%})")
    ax.set_ylabel(f"PC2 ({evr[1]:.1%})")
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=7)

plt.tight_layout()
plt.show()

print(
    "Doc bieu do: cum tach ro trong PCA-space (it chong lan mau) la bang chung manh hon "
    "bieu do location_x/y ve viec K-Means thuc su phan tach duoc du lieu, vi day dung dung "
    "khong gian 9-chieu (da scale) ma thuat toan dang lam viec, khong phai 2 truc vi tri vat ly."
)

In [ ]:
# Luu lai nhan cum ca 3 nhanh de tham khao/so sanh them o Task 6 neu can
cluster_output = df[["event_id", "match_id", "season_id", "team_name", "player_name"]].copy()
name_to_col = {
    "Baseline (co body_part)": "baseline",
    "Khong body_part": "no_body_part",
    "Body_part giam trong so": "weighted_body_part",
}
for name, suffix in name_to_col.items():
    cluster_output[f"cluster_id_{suffix}"] = labels_at_final_k[name]

out_path = f"{DATA_DIR}/shot_clusters_v4_k{FINAL_K}.csv"
cluster_output.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Da luu: {out_path}")
cluster_output.head(5)

## Kết luận

- **5v4.8 (purity) là bằng chứng chính** để trả lời câu hỏi gốc: nếu `Khong body_part` hoặc
  `Body_part giam trong so` có purity thấp hơn rõ rệt so với `Baseline`, nghĩa là 2 cách sửa đã
  giải quyết đúng vấn đề "cụm = bản sao body_part" phát hiện ở `kmeans_v3.ipynb`.
- Nếu `Khong body_part` cho silhouette **cao hơn** `Baseline` ở cùng k, đó là bằng chứng cụ thể
  rằng body_part đang **gây nhiễu** chứ không phải mang thêm tín hiệu — nên loại khỏi bước cluster,
  đưa `body_part` trở lại ở bước phân tích hậu-kỳ (mô tả từng cụm dùng `outcome`/`statsbomb_xg`/
  `body_part` như context, không phải feature đầu vào).
- Nếu `Body_part giam trong so` cho kết quả cân bằng hơn (silhouette tốt hơn Baseline, purity thấp
  hơn Baseline nhưng vẫn giữ được phần nào tín hiệu kỹ thuật dứt điểm), đây có thể là lựa chọn
  trung dung tốt hơn xoá hẳn — nhưng `BODY_PART_WEIGHT` là một hyperparameter cần ghi rõ lý do
  chọn giá trị (0.3) trong ADR nếu mang sang pipeline chính thức, vì đây không phải phép biến đổi
  chuẩn (scaler/log) mà là một lựa chọn thiết kế thủ công.
- Dù chọn nhánh nào, **silhouette tuyệt đối vẫn có thể ở mức thấp** (0.15–0.35) — đúng như phân
  tích ở `kmeans_v3.ipynb`: dữ liệu sút bóng biến thiên liên tục, không có ranh giới cụm sắc nét.
  Tiêu chí chọn cuối cùng nên là **centroid có diễn giải chiến thuật hợp lý** (đề xuất 3), không
  phải chỉ số nào cao nhất.